# 7.8 補了 PAD 該取哪個輸出？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：補齊長度像空椅子：不是新客人**

attention 別讀 PAD，loss 也別考 PAD。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/padding.svg")))

**先想一想：**長度2和4同batch補成4，短例子該取第幾格？

右PAD時，序列最後一格通常不是最後有效token。每筆各自找最後True位置，才能取下一字分數；不能一律logits[:,-1]。

**把直覺寫成數學：**最後有效索引=max{i:valid_i=True}；左PAD不能用sum(valid)-1。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
valid = torch.tensor([[True, True, False, False], [False, True, True, True]])
positions = torch.arange(4)[None].expand_as(valid)
last = positions.masked_fill(~valid, -1).amax(-1)
logits = torch.randn(2, 4, 5)
selected = logits[torch.arange(2), last]
print(last, selected.shape)
assert last.tolist() == [1, 3]

**如何讀結果：**這裏只示範取值；提供的簡單generate入口無padding，避免隱藏複雜度。

**只改一件事：**只把第一筆改成左PAD排列。
